# 01 — Exploratory Data Analysis (EDA)

Explore all three raw datasets (Resumes, Naukri, LinkedIn) before modelling.

In [ ]:
import sys
sys.path.insert(0, "..")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-dark")
%matplotlib inline

## 1. Resumes Dataset

In [ ]:
from src.data.load_data import load_resumes
resumes = load_resumes()
print(resumes.shape)
print(resumes["Category"].value_counts())

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
counts = resumes["Category"].value_counts()
sns.barplot(x=counts.values, y=counts.index, palette="magma", ax=ax)
ax.set_title("Resume Category Distribution")
plt.tight_layout()
plt.savefig("../reports/figures/resume_category_dist.png", dpi=150)
plt.show()

## 2. Naukri Jobs Dataset

In [ ]:
from src.data.load_data import load_naukri
naukri = load_naukri()
print(naukri.shape)
print(naukri.head(3))

In [ ]:
# Top job titles
top_titles = naukri["title"].value_counts().head(20)
fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(x=top_titles.values, y=top_titles.index, palette="viridis", ax=ax)
ax.set_title("Top 20 Job Titles — Naukri")
plt.tight_layout()
plt.savefig("../reports/figures/naukri_top_titles.png", dpi=150)
plt.show()

In [ ]:
# Top skills from Naukri
skill_counts = Counter()
for skills_str in naukri["skills"].dropna():
    for s in str(skills_str).split(","):
        s = s.strip().lower()
        if s:
            skill_counts[s] += 1
top_skills = pd.Series(skill_counts).sort_values(ascending=False).head(25)
fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(x=top_skills.values, y=top_skills.index, palette="crest", ax=ax)
ax.set_title("Top 25 In-Demand Skills — Naukri")
plt.tight_layout()
plt.savefig("../reports/figures/naukri_top_skills.png", dpi=150)
plt.show()

## 3. LinkedIn Jobs Dataset

In [ ]:
from src.data.load_data import load_linkedin
linkedin = load_linkedin()
print(linkedin.shape)
print(linkedin.head(3))

In [ ]:
# LinkedIn experience level distribution
fig, ax = plt.subplots(figsize=(10, 4))
linkedin["experience"].value_counts().head(10).plot(kind="bar", ax=ax, color="#6366f1")
ax.set_title("LinkedIn Experience Level Distribution")
ax.set_xlabel("")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("../reports/figures/linkedin_experience.png", dpi=150)
plt.show()

## 4. Resume Text Length Analysis

In [ ]:
resumes["word_count"] = resumes["Resume"].str.split().str.len()
resumes["char_count"] = resumes["Resume"].str.len()
print(resumes[["word_count", "char_count"]].describe())
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
resumes["word_count"].hist(bins=50, ax=axes[0], color="#6366f1")
axes[0].set_title("Word Count per Resume")
resumes["char_count"].hist(bins=50, ax=axes[1], color="#a855f7")
axes[1].set_title("Character Count per Resume")
plt.tight_layout()
plt.savefig("../reports/figures/resume_length.png", dpi=150)
plt.show()

## 5. Build & Save All Processed Artefacts

This is equivalent to running `python -m src.data.preprocess` from the project root.

Outputs:
- `data/processed/resumes_clean.csv`
- `data/interim/job_corpus.csv`
- `data/processed/jobs_clean.csv`

In [ ]:
from src.data.preprocess import preprocess_resumes, build_job_corpus

# 1. Resumes
resumes_proc = preprocess_resumes(resumes, save=True)
print("resumes_clean.csv saved:", resumes_proc.shape)

# 2. Jobs — Naukri + LinkedIn auto-detected
job_corpus = build_job_corpus(save=True)
print("job_corpus rows:  ", len(job_corpus))
print("clean_text ready: ", "clean_text" in job_corpus.columns)

In [ ]:
# Quick sanity check
print("Corpus columns:", list(job_corpus.columns))
print()
print(job_corpus[["title", "company", "experience", "skills"]].head(5).to_string())

## ✅ EDA complete. Next → **02_resume_classifier.ipynb**